In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import polars as pl
import numpy as np
import colorir as cl
from pathlib import Path
from tqdm import tqdm
from analyses import io

In [ ]:
cellveldf = (
    io
        .read_celldfs("../runs/speed/")
        .with_columns(
            replica=pl.col("replica").cast(pl.UInt32),
            gamma=20 - pl.col("energy").str.replace("energy-", "").cast(pl.UInt32),
        )
        .sort(["gamma", "replica", "wtime"])
)
cellveldf

In [ ]:
grouppers = ["gamma", "replica", "wtime"]
displdf = cellveldf.group_by(grouppers).agg(
    cluster_x=pl.col("center_x").mean(),
    cluster_y=pl.col("center_y").mean(),
    mean_displ=pl.col("displ").mean(),
    med_displ=pl.col("displ").median()
).sort(grouppers)
displdf

In [ ]:
max_time = displdf["wtime"].max()
stdf = displdf\
    .group_by(["gamma"])\
    .agg(
        stime=pl.col("wtime")\
            .filter(
                pl.col("mean_displ") <= pl.col("mean_displ").filter(pl.col("wtime") > 5e6).mean()
            )\
            .min(),
        ptime=pl.col("wtime")\
            .filter(
                pl.col("mean_displ") <= 180
            )\
            .min().fill_null(50e4)
    ).with_columns(atime=pl.when(pl.col("gamma") == 0).then(50e4).otherwise(2e6))
stdf

In [ ]:
# These sims should contain a single cell!!
sims = {}
for sp in Path("../runs/msd_2000_far/").iterdir():
    sim_type = sp.name
    gamma = 20 - int(sim_type.split("-")[1])
    sims[gamma] = pl.read_parquet(sp / "**" / "cells" / "*.parquet", include_file_paths="path", low_memory=True)

celldf = (
    pl
        .concat([sim.with_columns(gamma=k) for k, sim in sims.items()])
        .with_columns(list=pl.col("path").str.split("/"))
        .with_columns(
            wtime=pl.col("list").list.get(-1).str.replace(".parquet", "").cast(pl.UInt32),
            replica=pl.col("list").list.get(-3).cast(pl.UInt32),
        )
        .sort(["gamma", "replica", "time"])
)
assert (celldf["index"] == 0).all()
celldf

In [ ]:
cuts = pl.DataFrame({
    "dt": np.linspace(
        0,
        5e6,
        21
    )
})
    
trajdf = celldf\
    .filter(
        pl.col("time") % 100 == 0,
        gamma=0
    )\
    .select(
        "time",
        "center_x",
        "center_y",
        "replica"
    )\
    .sort(["replica", "time"])\
    .join(cuts, how="cross")\
    .filter(pl.col("time") <= pl.col("dt"))
trajdf

In [ ]:
format_kwargs = dict(
    template="plotly_white",
    xaxis_range=[1500, 2000], 
    yaxis_range=[1500, 2000],
    width=500, 
    height=500,
    xaxis_title="position x",
    yaxis_title="position y",
    yaxis_scaleanchor="x",
    yaxis_scaleratio=1,
    xaxis_constrain="domain"
)
px.line(
    trajdf, 
    x="center_x", 
    y="center_y",
    color="replica",
    animation_frame="dt"
).update_traces(
    visible='legendonly'
).update_layout(**format_kwargs)

In [ ]:
fig = px.line(
    trajdf.filter(dt=25e4), 
    x="center_x", 
    y="center_y",
    color="replica"
).update_layout(
    **format_kwargs,
    showlegend=False
)
io.save_plot(fig, "../plots/traj_gamma0_25e4")

In [ ]:
import math
def round_step_size(quantity, step_size) -> float:
    """Rounds a given quantity to a specific step size
    :param quantity: required
    :param step_size: required
    :return: decimal
    """
    precision: int = int(round(-math.log(step_size, 10), 0))
    return float(round(quantity, precision))

In [ ]:
dts = np.geomspace(
    10,
    2_000_000,
    100
)
dts = np.array([round_step_size(dt, 10) for dt in dts])
dts = np.unique(dts)  # Remove points that were rounded to the same number
# All numbers must be divisible by 10
assert np.equal(np.mod(dts / 10, 1), 0).all()
print(len(dts))
dts

In [ ]:
msddfs = []
groupped = celldf.filter(pl.col("time") > 5e5).sort("time").group_by(["gamma", "replica"])

for dt in tqdm(dts):
    n = dt // 10
    dx = pl.col("center_x").diff(n=n)
    dy = pl.col("center_y").diff(n=n)
    sd = (dx**2 + dy**2)

    msd = (
        groupped
        .agg(
            len=dx.drop_nulls().len(),
            msd=sd.mean(),
        )
        .with_columns(dt=dt)
        .filter(
            pl.col("len") >= 10,
            (pl.col("gamma") != 0) | (pl.col("dt") < 5e5)
        )
    )

    msddfs.append(msd)

msddf = pl.concat(msddfs)
msddf

In [ ]:
aggdf = msddf.filter(pl.col("gamma").is_in([0, 12])).group_by(["gamma", "dt"]).agg(
    mean=pl.col("msd").mean(),
    med=pl.col("msd").median(),
    min=pl.col("msd").min(),
    max=pl.col("msd").max(),
    std1=pl.col("msd").mean() + pl.col("msd").std(),
    std2=pl.col("msd").mean() - pl.col("msd").std()
).sort(["gamma", "dt"]).with_columns(
    pl.exclude("gamma").log().name.prefix("ln_")
)
aggdf

In [ ]:
palette = cl.StackPalette().load("carnival")
grad = cl.PolarGrad(palette, domain=[0, 20])
palette

In [ ]:
fig = go.Figure()
for gamma, group in aggdf.group_by("gamma"):
    gamma = gamma[0]
    color = grad(gamma)
    fig.add_traces([
        go.Scatter(
            x=group["dt"],
            y=group["mean"],
            mode="lines",
            line_color=color,
            name=f"gamma-{gamma}",
            legendgroup=gamma
        ),
        go.Scatter(
            x=pl.concat([group["dt"], group["dt"][::-1]]),
            y=pl.concat([group["min"], group["max"][::-1]]),
            mode="lines",
            line_color="rgba(0, 0, 0, 0)",
            fillcolor=color,
            opacity=0.3,
            fill="toself",
            hoverinfo="skip",
            showlegend=False,
            legendgroup=gamma
        )
    ])
fig.update_layout(
    template="plotly_white",
    width=500,
    height=500,
    xaxis_title="Δt",
    yaxis_title="MSD"
)
io.save_plot(fig, "../plots/msd")

In [ ]:
fig2 = go.Figure(fig.update_layout(
    xaxis_type="log",
    yaxis_type="log",
    width=500,
    height=500,
    xaxis_title="log(Δt)",
    yaxis_title="log(MSD)"
))
fig3 = go.Figure()
for gamma, group in aggdf.group_by("gamma"):
    gamma = gamma[0]
    poly = np.polynomial.Polynomial.fit(x=group["ln_dt"], y=group["ln_mean"], deg=5).convert()
    fig2.add_trace(go.Scatter(
        x=group["dt"],
        y=np.exp(poly(group["ln_dt"])),
        mode="lines",
        line_dash="dash",
        line_color="#000000",
        showlegend=False,
        legendgroup=gamma
    ))

    # Resampling here is slightly dangerous as the polynomial function can hallucinate between points
    # Should be fine as long as the initial sample is not too little
    log_sample = np.log(np.geomspace(
        500,
        50e4 if gamma == 0 else 2_000_000,
        2000
    ))
    # If the sample size is big enough you can just use that
    log_sample = group["ln_dt"]
    
    first_deriv = poly.deriv()
    fig3.add_traces([
        go.Scatter(
            x=group["dt"],
            y=np.gradient(group["ln_mean"], group["ln_dt"]),
            line_color=grad(gamma),
            name="gamma-" + str(gamma),
            legendgroup=gamma
        ),
        go.Scatter(
            x=np.concat([group["dt"], group["dt"][::-1]]),
            y=np.concat([
                np.gradient(group["ln_std1"], group["ln_dt"]),
                np.gradient(group["ln_std2"], group["ln_dt"])[::-1],
            ]),
            mode="lines",
            line_color="rgba(0, 0, 0, 0)",
            fillcolor=grad(gamma),
            opacity=0.3,
            fill="toself",
            hoverinfo="skip",
            showlegend=False,
            legendgroup=gamma
        ),
        go.Scatter(
            x=np.exp(log_sample),
            y=first_deriv(log_sample),
            mode="lines",
            line_dash="dash",
            line_color="#000000",
            showlegend=False,
            legendgroup=gamma
        )
    ])
io.save_plot(fig2, "../plots/log_log")
fig2.show()
fig3.update_layout(
    template="plotly_white",
    xaxis_type="log",
    width=600,
    height=400,
    xaxis_title="log(Δt)",
    yaxis_title="diffusive exponent",
    yaxis_range=[0, 2]
)
io.save_plot(fig3, "../plots/diffusive exponent_data_model")